In [0]:
from pyspark.sql import types as T
from pyspark.sql import functions as F

In [0]:
catalog_name = 'ecommerce'

In [0]:
df_products = spark.table(f'{catalog_name}.silver.sil_products')
df_brands = spark.table(f'{catalog_name}.silver.sil_brands')
df_category = spark.table(f'{catalog_name}.silver.sil_category')

In [0]:
df_products.createOrReplaceTempView('v_products')
df_brands.createOrReplaceTempView('v_brands')
df_category.createOrReplaceTempView('v_category')

In [0]:
spark.sql('select * from v_products limit 5').display()


In [0]:
spark.sql('select * from v_category limit 5').display()

In [0]:
spark.sql('select * from v_brands limit 5').display()

In [0]:
spark.sql(f"use catalog {catalog_name}")

In [0]:
%sql

create or replace table gold.gld_dim_products as

with brands_category as (
    select 
        vc.category_code,
        vc.category_name,
        vb.brand_code,
        vb.brand_name 
    from v_brands vb
    inner join v_category vc
    on vb.category_code = vc.category_code
)
select
    vp.product_id,
    vp.sku,
    vp.category_code,
    coalesce(bc.category_name,'not available') as category_name,
    vp.brand_code,
    coalesce(bc.brand_name,'not available') as brand_name,
    coalesce(vp.color,'not available') as color,
    vp.size,
    vp.material,
    vp.weight_grams,
    vp.length_cm,
    vp.width_cm,
    vp.height_cm,
    vp.rating_count,
    vp._source_file as file_name,
    vp.injestedAt as time_stamp

from v_products vp
left join brands_category bc
on vp.brand_code = bc.brand_code




In [0]:
%sql
select * from gold.gld_dim_products

In [0]:
# India states
india_region = {
    "MH": "West",
    "GJ": "West",
    "RJ": "West",
    "KA": "South",
    "TN": "South",
    "TS": "South",
    "AP": "South",
    "KL": "South",
    "UP": "North",
    "WB": "North",
    "DL": "North"
}

# Australia states
australia_region = {
    "VIC": "SouthEast",
    "WA": "West",
    "NSW": "East",
    "QLD": "NorthEast"
}

# United Kingdom states
uk_region = {
    "ENG": "England",
    "WLS": "Wales",
    "NIR": "Northern Ireland",
    "SCT": "Scotland"
}

# United States states
us_region = {
    "MA": "NorthEast",
    "FL": "South",
    "NJ": "NorthEast",
    "CA": "West",
    "NY": "NorthEast",
    "TX": "South"
}

# UAE states
uae_region = {
    "AUH": "Abu Dhabi",
    "DU": "Dubai",
    "SHJ": "Sharjah"
}

# Singapore
singapore_region = {
    "SG": "Singapore"
}

# Canada states
canada_region = {
    "BC": "West",
    "AB": "West",
    "ON": "East",
    "QC": "East",
    "NS": "East",
    "IL": "Other"
}

# Combine into a master dictionary
country_state_map = {
    "India": india_region,
    "Australia": australia_region,
    "United Kingdom": uk_region,
    "United States": us_region,
    "United Arab Emirates": uae_region,
    "Singapore": singapore_region,
    "Canada": canada_region
}

In [0]:
country_state_map

In [0]:
from pyspark.sql import Row
rows = []
for country,states in country_state_map.items():
    for state,region in states.items():
        rows.append(Row(country = country,state = state,region = region))
rows[:10]

In [0]:
df_region_map = spark.createDataFrame(rows)

In [0]:
display(df_region_map.limit(5))

In [0]:
df_cust_silver = spark.table(f'{catalog_name}.silver.sil_customers')
df_cust_silver.show()

In [0]:
df_gold_customers = df_cust_silver.join(df_region_map,on=['country','state'],how ='left')
df_gold_customers = df_gold_customers.fillna('other',subset=['region'])

In [0]:
df_gold_customers.display()

In [0]:
df_gold_customers = df_gold_customers.withColumnRenamed('_source_file','file_name')

In [0]:
col_names_rearrange_cust = ['customer_id','phone','country','country_code','state','region','file_name','injestedAt']
df_gold_customers = df_gold_customers.select(col_names_rearrange_cust)
df_gold_customers.show(truncate=False)

In [0]:
df_gold_customers.write\
    .format('delta')\
    .mode('overwrite')\
    .saveAsTable(f'{catalog_name}.gold.gld_customers')

###date


In [0]:
df_date_silver = spark.table(f'{catalog_name}.silver.sil_date')
df_date_silver.display()

In [0]:
df_gold_date = df_date_silver.withColumnRenamed('_source_file','file_name')

In [0]:
df_gold_date = df_gold_date.withColumn('month_name',F.date_format(F.col('date'),'MMMM'))


In [0]:
df_gold_date = df_gold_date.withColumn(
    'is_weekend',
    F.when(F.col('day_name').isin(['Saturday','Sunday']),1).otherwise(0)
)

In [0]:
df_gold_date.show(truncate=False)

In [0]:
df_gold_date = df_gold_date.withColumn('date_id',F.date_format(F.col('date'),"yyyyMMdd").cast('int'))

In [0]:
df_gold_date.show(truncate=False)

In [0]:
desired_table_order = ['date_id','date','year','month_name','day_name','is_weekend','quarter','week','file_name','injestedAt']
df_gold_date = df_gold_date.select(desired_table_order)
display(df_gold_date)

In [0]:
df_gold_date.write\
    .format('delta')\
    .mode('overwrite')\
    .option('mergeSchema',True)\
    .saveAsTable(f'{catalog_name}.gold.gld_date')

In [0]:
df_gold_customers.show()